In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# 1. INITIALIZE GOLD SESSION
if 'spark' in locals(): spark.stop()

spark = SparkSession.builder \
    .appName("Prism-Risk-Gold-Simulation") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "sentinel_password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("🚀 Gold Engine Online. Loading Silver Data...")

try:
    # 2. LOAD REFINED DATA
    df_silver = spark.read.format("delta").load("s3a://silver/transactions_refined")
    
    # 3. EXTRACT RISK POPULATIONS (Optimization)
    # Instead of pulling 10M rows to Python, we just pull the counts of each risk bucket.
    # This is O(1) complexity for the simulation instead of O(N).
    print("📊 Aggregating Risk Profiles...")
    risk_counts = df_silver.groupBy("preliminary_risk_score").count().collect()
    
    # Convert to a dictionary for easy access: {0.95: 15021, 0.40: 199753, ...}
    risk_profile = {row['preliminary_risk_score']: row['count'] for row in risk_counts}
    
    count_high = risk_profile.get(0.95, 0)
    count_med  = risk_profile.get(0.40, 0)
    
    print(f"   - High Risk Entities (0.95): {count_high:,}")
    print(f"   - Medium Risk Entities (0.40): {count_med:,}")

    # 4. MONTE CARLO SIMULATION (Binomial Method)
    # Scenario:
    # - High Risk (0.95) = Sanctions Violation = $50,000 Fine
    # - Med Risk (0.40) = Smurfing Suspicion = $10,000 Fine if proven
    
    NUM_SIMULATIONS = 5000
    FINE_HIGH = 50000
    FINE_MED = 10000
    
    print(f"\n🎲 Running {NUM_SIMULATIONS} Monte Carlo scenarios...")
    start_sim = time.time()
    
    results = []
    
    for _ in range(NUM_SIMULATIONS):
        # We use a Binomial distribution: "Flip a coin N times with probability P"
        # This is mathematically identical to iterating rows but instantaneous.
        
        # Simulating confirmed sanctions hits
        confirmed_sanctions = np.random.binomial(n=count_high, p=0.95)
        
        # Simulating confirmed smurfing cases (lower probability of conversion to fine)
        # Let's assume only 5% of suspicious smurfing actually results in a fine
        confirmed_smurfing = np.random.binomial(n=count_med, p=0.05) 
        
        total_fine = (confirmed_sanctions * FINE_HIGH) + (confirmed_smurfing * FINE_MED)
        results.append(total_fine)
        
    print(f"✅ Simulation Complete in {round(time.time() - start_sim, 2)}s")

    # 5. CALCULATE VALUE AT RISK (VaR)
    results = np.array(results)
    mean_exposure = np.mean(results)
    var_95 = np.percentile(results, 95) # 95% Confidence Interval
    var_99 = np.percentile(results, 99) # 99% Confidence Interval (Stress Test)

    # 6. EXECUTIVE REGULATORY REPORT
    print("\n" + "="*60)
    print("🛡️ PRISM-RISK SENTINEL: REGULATORY CAPITAL REPORT")
    print("="*60)
    print(f"✅ Total Transactions Scanned: {df_silver.count():,}")
    print(f"🚨 Sanctions Hits (Direct & Evasive): {count_high:,}")
    print(f"⚠️ Potential Smurfing Attempts: {count_med:,}")
    print("-" * 60)
    print("💰 PROJECTED REGULATORY FINES (Probabilistic Model):")
    print(f"   • Expected Mean Loss:      ${mean_exposure:,.2f}")
    print(f"   • VaR (95% Confidence):    ${var_95:,.2f}")
    print(f"   • VaR (99% Stress Test):   ${var_99:,.2f}")
    print("-" * 60)
    print("📝 STRATEGIC RECOMMENDATION:")
    
    if var_95 > 100000000: # $100M Threshold
        print("🔴 CRITICAL: Insolvency Risk. Immediate capital injection required.")
    elif var_95 > 10000000: # $10M Threshold
        print("🟠 WARNING: Reserve Capital insufficient. Increase provisions by 15%.")
    else:
        print("🟢 STABLE: Current capital reserves cover 99% of risk scenarios.")
    print("="*60)

    # 7. VISUALIZATION
    plt.figure(figsize=(12, 6))
    plt.hist(results, bins=50, color='#2c3e50', alpha=0.7, edgecolor='black')
    plt.axvline(mean_exposure, color='gold', linestyle='--', linewidth=2, label=f'Expected: ${mean_exposure/1e6:.1f}M')
    plt.axvline(var_95, color='red', linestyle='-', linewidth=2, label=f'VaR 95%: ${var_95/1e6:.1f}M')
    
    plt.title('Monte Carlo Simulation: Regulatory Fine Exposure Distribution', fontsize=14)
    plt.xlabel('Total Estimated Fines ($)', fontsize=12)
    plt.ylabel('Frequency (Likelihood)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

except Exception as e:
    print(f"❌ Simulation Failed: {e}")